In [1]:
import os, sys
from tqdm import tqdm
import torch
import numpy as np
from scipy import stats
from procrustes import rotational
import matplotlib.pyplot as plt

from cmm.ffxml import ForceFieldXML
from cmm.topology import Topology
from cmm.units import BOHR2NM, BOHR2ANG, HARTREE2KCAL, AMU2ELECTRON_MASS, HARTREE2WAVENUMBER
from cmm.drivers import OptimizationDriver, HarmonicAnalysisDriver
from cmm.misc_utils import read_xyz, write_xyz, get_masses

import openmm.app as app
from ase.io import read

In [2]:
def rmsd(A: np.ndarray, B: np.ndarray):
    return np.sqrt(np.mean(np.sum((A - B)**2, axis=1)))

In [3]:
torch.set_default_dtype(torch.float64)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
ff_path = os.path.join(os.path.abspath(''), '../scripts/ion_water_params_8_27.xml')
ff = ForceFieldXML(ff_path, device=device)

In [4]:
water_cluster_pdb_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.pdb')
water_cluster_xyz_path = os.path.join(os.path.abspath(''), '../tests/data/water_clusters/all_reference_clusters.xyz')

f_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_f_scan.pdb')
cl_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_cl_scan.pdb')
br_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_br_scan.pdb')
i_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_i_scan.pdb')

li_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_li_scan.pdb')
na_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_na_scan.pdb')
k_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_k_scan.pdb')
rb_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_rb_scan.pdb')
cs_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_cs_scan.pdb')

mg_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_mg_scan.pdb')
ca_water_scan_pdb_path = os.path.join('/home/heindelj/research/Teresa/reactive_force_fields/reference_data/CMM_Data/ion_water/ion_water_scans/h2o_ca_scan.pdb')

In [5]:
water_cluster_pdb = app.PDBFile(water_cluster_pdb_path)
positions = [water_cluster_pdb.getPositions(True, frame=i)._value * 10.0 for i in range(water_cluster_pdb.getNumFrames())]

topologies = [Topology.fromMultiPDB(water_cluster_pdb_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies]

topologies_no_fd = [Topology.fromMultiPDB(water_cluster_pdb_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems_no_fd = [ff.parametrize(topology, use_fd_morse=False, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies_no_fd]

topologies_with_fd = [Topology.fromMultiPDB(water_cluster_pdb_path, device, frame_index=i) for i in range(water_cluster_pdb.getNumFrames())]
systems_with_fd = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False) for topology in topologies_with_fd]

reference_coords, reference_labels = read_xyz(water_cluster_xyz_path)
reference_masses = [get_masses(reference_labels[i]) for i in range(len(reference_labels))]

/home/heindelj/miniforge3/envs/pycmm/lib/python3.12/site-packages/torch/nested/__init__.py:226: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  return _nested.nested_tensor(


In [6]:
def create_systems(pdb_path: str):
    pdb_openmm = app.PDBFile(pdb_path)
    positions = [pdb_openmm.getPositions(True, frame=i)._value * 10.0 for i in range(pdb_openmm.getNumFrames())]
    topologies = [Topology.fromMultiPDB(pdb_path, device, frame_index=i) for i in range(pdb_openmm.getNumFrames())]
    systems = [ff.parametrize(topology, use_fd_morse=True, periodic=False, use_lr_dispersion=False, use_cutoff=False, use_ewald=False) for topology in topologies]
    return positions, systems

In [7]:
f_water_positions, f_water_scan_systems = create_systems(f_water_scan_pdb_path)
cl_water_positions, cl_water_scan_systems = create_systems(cl_water_scan_pdb_path)
br_water_positions, br_water_scan_systems = create_systems(br_water_scan_pdb_path)
i_water_positions, i_water_scan_systems = create_systems(i_water_scan_pdb_path)

li_water_positions, li_water_scan_systems = create_systems(li_water_scan_pdb_path)
na_water_positions, na_water_scan_systems = create_systems(na_water_scan_pdb_path)
k_water_positions, k_water_scan_systems = create_systems(k_water_scan_pdb_path)
rb_water_positions, rb_water_scan_systems = create_systems(rb_water_scan_pdb_path)
cs_water_positions, cs_water_scan_systems = create_systems(cs_water_scan_pdb_path)

mg_water_positions, mg_water_scan_systems = create_systems(mg_water_scan_pdb_path)
ca_water_positions, ca_water_scan_systems = create_systems(ca_water_scan_pdb_path)

In [8]:
ions_systems_and_positions = [
    ("li+", li_water_positions[7], li_water_scan_systems[7]),
    ("na+", na_water_positions[7], na_water_scan_systems[7]),
    ("k+", k_water_positions[7], k_water_scan_systems[7]),
    ("rb+", rb_water_positions[7], rb_water_scan_systems[7]),
    ("cs+", cs_water_positions[7], cs_water_scan_systems[7]),
    ("mg2+", mg_water_positions[7], mg_water_scan_systems[7]),
    ("ca2+", ca_water_positions[7], ca_water_scan_systems[7]),
    ("f-", f_water_positions[7], f_water_scan_systems[7]),
    ("cl-", cl_water_positions[7], cl_water_scan_systems[7]),
    ("br-", br_water_positions[7], br_water_scan_systems[7]),
    ("i-", i_water_positions[7], i_water_scan_systems[7])
]

def optimize_ion_water_dimers_and_compute_frequencies(data_in):
    optimized_coords = []
    optimized_energies = []
    harmonic_frequencies = []
    for i in range(len(data_in)):
        label, positions, system = data_in[i]
        opt_driver = OptimizationDriver(system, tolerance=1e-12)
        coords = torch.from_numpy(positions / BOHR2ANG).to(device).requires_grad_(False)
        box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
        coords_opt, box_opt, result = opt_driver.run(coords, box)
        energy = result.fun * HARTREE2KCAL
        print(f"Optimized H2O...{label}: E = {energy}")
        optimized_coords.append(coords_opt)
        optimized_energies.append(energy)
        
        harmonic_driver = HarmonicAnalysisDriver(system)
        masses = get_masses(system.top._atom_symbols)
        hessian = harmonic_driver.run(coords_opt.to(device).requires_grad_(False), box, masses * AMU2ELECTRON_MASS)
        eigvals, eigvecs = torch.linalg.eigh(hessian)
        freqs = torch.sqrt(eigvals) * HARTREE2WAVENUMBER
        nan_mask = torch.isnan(freqs)
        nan_indices = torch.nonzero(nan_mask)
        freqs[nan_indices] = torch.zeros(nan_indices.size())
        harmonic_frequencies.append(freqs)

    return optimized_coords, optimized_energies, harmonic_frequencies

In [9]:
optimized_ion_water_coords, optimized_ion_water_energies, ion_water_dimer_freqs = optimize_ion_water_dimers_and_compute_frequencies(ions_systems_and_positions)

Optimized H2O...li+: E = -34.7870986026486
Optimized H2O...na+: E = -24.215880741400337
Optimized H2O...k+: E = -17.579489311939977
Optimized H2O...rb+: E = -15.402716035561792
Optimized H2O...cs+: E = -13.873209834558134
Optimized H2O...mg2+: E = -85.03657747808252
Optimized H2O...ca2+: E = -59.047459185993176
Optimized H2O...f-: E = -29.95605073497724
Optimized H2O...cl-: E = -15.392789742590574
Optimized H2O...br-: E = -13.43423572919271
Optimized H2O...i-: E = -11.245682465073177


In [10]:
ion_water_dimer_freqs

[tensor([0.0000e+00, 1.2505e-05, 4.2189e-05, 8.6745e+01, 1.0929e+02, 1.5461e+02,
         3.6583e+02, 6.7410e+02, 6.8225e+02, 1.7347e+03, 3.7640e+03, 3.8526e+03]),
 tensor([0.0000e+00, 0.0000e+00, 1.6351e-05, 3.6865e+01, 6.5155e+01, 6.5776e+01,
         2.6491e+02, 3.4748e+02, 5.2618e+02, 1.7196e+03, 3.7822e+03, 3.8751e+03]),
 tensor([0.0000e+00, 1.3214e-05, 2.8302e-05, 2.4644e+01, 4.0783e+01, 5.4075e+01,
         2.3428e+02, 2.8711e+02, 4.5224e+02, 1.7043e+03, 3.7909e+03, 3.8871e+03]),
 tensor([0.0000e+00, 8.2163e-06, 2.1678e-05, 1.9420e+01, 3.1777e+01, 4.9774e+01,
         1.9081e+02, 2.8941e+02, 4.2014e+02, 1.6988e+03, 3.7944e+03, 3.8919e+03]),
 tensor([0.0000e+00, 6.9550e-06, 9.9763e-06, 1.7251e+01, 2.7893e+01, 4.6914e+01,
         1.6514e+02, 2.8597e+02, 4.0703e+02, 1.6952e+03, 3.7969e+03, 3.8953e+03]),
 tensor([0.0000e+00, 0.0000e+00, 1.4084e-05, 1.4038e+02, 1.8505e+02, 2.6110e+02,
         8.2966e+02, 8.7367e+02, 1.1037e+03, 1.8487e+03, 3.6336e+03, 3.6984e+03]),
 tensor([   0.00

In [11]:
optimized_ion_water_coords[8] * BOHR2ANG

tensor([[ 0.0186, -0.3729, -0.3439],
        [ 0.7350, -0.1420,  0.2450],
        [ 0.4701, -1.0340, -0.9179],
        [ 1.9443, -2.3692, -1.8273]])

In [12]:
li_rmsd = rmsd(optimized_ion_water_coords[0].cpu().numpy() * BOHR2ANG, li_water_positions[7])
na_rmsd = rmsd(optimized_ion_water_coords[1].cpu().numpy() * BOHR2ANG, na_water_positions[7])
k_rmsd = rmsd(optimized_ion_water_coords[2].cpu().numpy() * BOHR2ANG, k_water_positions[7])
rb_rmsd = rmsd(optimized_ion_water_coords[3].cpu().numpy() * BOHR2ANG, rb_water_positions[7])
cs_rmsd = rmsd(optimized_ion_water_coords[4].cpu().numpy() * BOHR2ANG, cs_water_positions[7])

mg_rmsd = rmsd(optimized_ion_water_coords[5].cpu().numpy() * BOHR2ANG, mg_water_positions[7])
ca_rmsd = rmsd(optimized_ion_water_coords[6].cpu().numpy() * BOHR2ANG, ca_water_positions[7])

f_rmsd = rmsd(optimized_ion_water_coords[7].cpu().numpy() * BOHR2ANG, f_water_positions[7])
cl_rmsd = rmsd(optimized_ion_water_coords[8].cpu().numpy() * BOHR2ANG, cl_water_positions[7])
br_rmsd = rmsd(optimized_ion_water_coords[9].cpu().numpy() * BOHR2ANG, br_water_positions[7])
i_rmsd = rmsd(optimized_ion_water_coords[10].cpu().numpy() * BOHR2ANG, i_water_positions[7])

In [13]:
ion_water_rmsds = np.array([
    li_rmsd, na_rmsd, k_rmsd, rb_rmsd, cs_rmsd, mg_rmsd, ca_rmsd, f_rmsd, cl_rmsd, br_rmsd, i_rmsd
])
print(ion_water_rmsds)

[0.10108878 0.06078998 0.06126312 0.06201026 0.06955013 0.16602235
 0.07452074 0.06273041 0.0542255  0.08558168 0.06161434]


In [14]:
coords = torch.from_numpy(positions[1] / BOHR2ANG).to(device).requires_grad_(False)
box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
energies = systems[1].getEnergy(coords, box)

for key in energies:
    print(f"{key}: {energies[key] * HARTREE2KCAL} kcal/mol")

bond: 0.35676326831702043 kcal/mol
angle: 0.023446173018395126 kcal/mol
torsion: 0.0 kcal/mol
bond_bond: -0.005611845671081472 kcal/mol
bond_angle: 0.03452899245356382 kcal/mol
angle_angle: 0.0 kcal/mol
torsion_bond: 0.0 kcal/mol
torsion_angle: 0.0 kcal/mol
torsion_angle_angle: 0.0 kcal/mol
perm_elec: -25.909409713076045 kcal/mol
pol: -3.7103559196676787 kcal/mol
ct_direct: -6.5340623624889105 kcal/mol
xpol: -0.6042721511840822 kcal/mol
pauli: 27.69132165343953 kcal/mol
disp: -6.1386630971369645 kcal/mol
total: -14.796315001996248 kcal/mol


In [15]:
opt_coords_path_with_fd = os.path.join(os.path.abspath(''), 'reference_opt_cmm_with_fd_morse.xyz')
coords_opt, labels = read_xyz(opt_coords_path_with_fd)

all_perm_elec = []
all_pauli = []
all_dispersion = []
all_induction = []
all_total = []

for i in range(len(coords_opt)):
    coords = torch.from_numpy(coords_opt[i] / BOHR2ANG).to(device).requires_grad_(False)
    box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
    energies = systems[i].getEnergy(coords, box)

    all_perm_elec.append(energies['perm_elec'].item() * HARTREE2KCAL)
    all_pauli.append(energies['pauli'].item() * HARTREE2KCAL)
    all_dispersion.append(energies['disp'].item() * HARTREE2KCAL)
    all_induction.append((energies['pol'] + energies['ct_direct']).item() * HARTREE2KCAL)
    all_total.append(energies['total'].item() * HARTREE2KCAL)
print(all_perm_elec)
print(all_pauli)
print(all_dispersion)
print(all_induction)
print(all_total)

[-8.633694107642922, -29.31755952908293, -53.336800150900466, -69.81120832622106, -84.11562801355973, -84.5391827784503, -87.45869492435784, -85.65131802694893, -107.69200471434621, -137.19733671059248, -138.01371115934057, -154.74666928512727, -177.19802761967046, -193.56126374472962, -307.871810891364, -299.2727046206214, -299.3387016620027, -310.2405759251226, -310.25852086389267, -322.22787462104156, -396.25743099100663, -382.63706752126467, -381.3793250318622, -380.97325979139526, -508.9299003481404]
[8.647684676093222, 33.816616952878384, 65.9160899055834, 88.21018325162515, 101.79320348283916, 103.57423303941361, 109.86896643004145, 108.19334333579047, 134.0032410841238, 170.29055373539066, 172.9728097782078, 194.34001718224047, 224.04717497652874, 245.61274905019624, 393.59378693069755, 376.3100323803461, 377.8977711600202, 396.15140641233705, 398.3206143227763, 413.01440005013563, 511.32515955234027, 486.9005702000113, 480.6109289408371, 488.87595089319865, 661.3870643774316]


In [16]:
for energy in all_total:
    print(energy)

-4.867140234258436
-15.206384408515312
-27.45512297728221
-36.257658531961674
-45.403132327678655
-45.24058827077652
-45.65036300139919
-44.873459049720566
-57.27975889817106
-72.64075929418897
-72.80904484960594
-82.32569631964932
-93.94691951789204
-103.2189809944901
-164.8676422012156
-163.43741904690776
-163.2236036804729
-164.74888698785244
-165.0137391864955
-175.57057031272387
-213.59611743986164
-209.6191973509433
-208.9049817328674
-201.4727850117892
-274.10426233992075


In [17]:
def optimize_all_reference_structures_and_compute_rmsds(systems, reference_coords):
    optimized_coords = []
    optimized_energies = []
    rmsds = []
    for i in range(len(systems)):
        opt_driver = OptimizationDriver(systems[i])
        coords = torch.from_numpy(reference_coords[i] / BOHR2ANG).to(device).requires_grad_(False)
        box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
        coords_opt, box_opt, result = opt_driver.run(coords, box)
        optimized_energies.append(result.fun * HARTREE2KCAL)
        if result.success == False:
            print(f"Warning: Optimization of structure {i} did not converge. Final energy was {optimized_energies[-1]} kcal/mol.")
        # Get RMSD and store new coordinates.
        # Align structures first in case they rotated during optimization.
        coords_opt = coords_opt.numpy() * BOHR2ANG
        result = rotational(coords_opt, reference_coords[i])
        optimized_coords.append(result.new_a)
        rmsd_i = rmsd(result.new_a, reference_coords[i])
        rmsds.append(rmsd_i)
        print(f"Structure {i}: Energy = {optimized_energies[-1]} kcal/mol, RMSD = {rmsd_i} Ang.")
    
    return optimized_coords, optimized_energies, rmsds

In [18]:
opt_coords, opt_energies, opt_rmsds = optimize_all_reference_structures_and_compute_rmsds(systems, reference_coords)

# Write out the optimized coords to a file
write_xyz("reference_opt_cmm_with_fd_morse.xyz", reference_labels, opt_coords)

Structure 0: Energy = -4.867140506747001 kcal/mol, RMSD = 0.036496749587889954 Ang.
Structure 1: Energy = -15.206384526724465 kcal/mol, RMSD = 0.06885408076683572 Ang.
Structure 2: Energy = -27.455122932676584 kcal/mol, RMSD = 0.05703519303808381 Ang.
Structure 3: Energy = -36.257658421089275 kcal/mol, RMSD = 0.05671862782914064 Ang.
Structure 4: Energy = -45.40313218144183 kcal/mol, RMSD = 0.04675692473954364 Ang.
Structure 5: Energy = -45.240588280736546 kcal/mol, RMSD = 0.0637896649618314 Ang.
Structure 6: Energy = -45.65036243517163 kcal/mol, RMSD = 0.06451870947241209 Ang.
Structure 7: Energy = -44.87345926296371 kcal/mol, RMSD = 0.06158054695052744 Ang.
Structure 8: Energy = -57.27975871579441 kcal/mol, RMSD = 0.05751719209184388 Ang.
Structure 9: Energy = -72.64075931362785 kcal/mol, RMSD = 0.05575698204667176 Ang.
Structure 10: Energy = -72.8090446496837 kcal/mol, RMSD = 0.056743572690585505 Ang.
Structure 11: Energy = -82.325696439944 kcal/mol, RMSD = 0.05255251390874921 Ang.


KeyboardInterrupt: 

In [ ]:
for val in opt_rmsds:
    print(val)

0.036496749587889954
0.06885408076683572
0.05703519303808381
0.05671862782914064
0.04675692473954364
0.0637896649618314
0.06451870947241209
0.06158054695052744
0.05751719209184388
0.05575698204667176
0.056743572690585505
0.05255251390874921
0.05185194354893609
0.07677682395968433
0.06366964845289753
0.08324005606917234
0.07347325902894286
0.08237898959205348
0.10050705833855154
0.06122448583251924
0.10215215231504562
0.06946205916800269
0.0822888884417537
0.06829356547297079
0.10299627552225075


In [ ]:
def compute_hbond_distance_freq_correlation_for_ref_clusters(systems, optimized_coord_file):
    all_dists = []
    all_freqs = []

    coords_opt, labels = read_xyz(optimized_coord_file)
    masses = [get_masses(labels[i]) for i in range(len(labels))]
    box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)
    for geom_index in tqdm(range(len(coords_opt))):
        box = torch.tensor(np.eye(3) * 100.0, requires_grad=False, device=device)

        harmonic_driver = HarmonicAnalysisDriver(systems[geom_index])
        hessian = harmonic_driver.run(torch.from_numpy(coords_opt[geom_index] / BOHR2ANG).to(device).requires_grad_(False), box, masses[geom_index] * AMU2ELECTRON_MASS)
        eigvals, eigvecs = torch.linalg.eigh(hessian)
        freqs = torch.sqrt(eigvals) * HARTREE2WAVENUMBER

        # Find the atom which moves most for each hbond frequency
        minimum_hbond_frequency = 2500.0
        maximum_hbond_frequency = 3800.0
        mask = (freqs >= minimum_hbond_frequency) & (freqs <= maximum_hbond_frequency)
        hbond_indices = torch.nonzero(mask).squeeze()
        hbond_eigvces = eigvecs[:, hbond_indices]
        if hbond_eigvces.dim() == 1:
            hbond_eigvces.unsqueeze_(1)
        hbond_freqs = freqs[hbond_indices]
        if hbond_freqs.ndim == 0:
            hbond_freqs = hbond_freqs[np.newaxis]

        h_indices = torch.zeros(hbond_eigvces.size(-1), dtype=torch.long)
        for i in range(hbond_eigvces.size(-1)):
            mode = hbond_eigvces[:, i].reshape(-1, 3)
            h_index = torch.argmax(torch.linalg.norm(mode, dim=1))
            h_indices[i] = h_index

        # Get the distances between each hydrogen and the other oxygens in the system
        all_O_indices = []
        all_H_indices = []
        for i_geom in range(len(labels)):
            O_indices = []
            H_indices = []
            for i, label in enumerate(labels[i_geom]):
                if label == 'O':
                    O_indices.append(i)
                if label == 'H':
                    H_indices.append(i)
            all_O_indices.append(O_indices)
            all_H_indices.append(H_indices)

        all_O_indices = [torch.tensor(all_O_indices[i]) for i in range(len(all_O_indices))]
        all_H_indices = [torch.tensor(all_H_indices[i]) for i in range(len(all_H_indices))]

        for index in h_indices:
            if reference_labels[geom_index][index] != 'H':
                print("Warning: Found an h-bond frequency where the most mobile atom was not hydrogen.")

        H_atoms_hbond_only = coords_opt[geom_index][h_indices]
        O_atoms = coords_opt[geom_index][all_O_indices[geom_index]]
        if H_atoms_hbond_only.ndim == 1:
            H_atoms_hbond_only = H_atoms_hbond_only[np.newaxis, :]
            
        dist_vecs = H_atoms_hbond_only[:, np.newaxis, :] - O_atoms[np.newaxis, :, :]
        OH_hbond_dists = np.array([np.min(np.linalg.norm(dist_vecs[i], axis=1)) for i in range(dist_vecs.shape[0])])
        #OH_hbond_dists_sorted = np.sort(OH_hbond_dists)[::-1]
        all_dists.append(OH_hbond_dists)#_sorted)
        all_freqs.append(hbond_freqs.numpy())
    return np.concatenate(all_dists), np.concatenate(all_freqs)

In [ ]:
#opt_coords_path_no_fd = os.path.join(os.path.abspath(''), 'reference_opt_cmm_without_fd_morse.xyz')
#all_dists_no_fd, all_freqs_no_fd = compute_hbond_distance_freq_correlation_for_ref_clusters(systems_no_fd, opt_coords_path_no_fd)
#
#opt_coords_path_with_fd = os.path.join(os.path.abspath(''), 'reference_opt_cmm_with_fd_morse.xyz')
#all_dists_with_fd, all_freqs_with_fd = compute_hbond_distance_freq_correlation_for_ref_clusters(systems_with_fd, opt_coords_path_with_fd)

In [ ]:
def plot_badger_rule_correlation(r_hbond_no_fd, freqs_hbond_no_fd,
                                 r_hbond_with_fd, freqs_hbond_with_fd,
                                 title="Badger Rule Correlation"):
    r_eq_cmm = 0.9589289
    freq_ave_cmm = (3945.054377 + 3834.691479) / 2

    r_eq_wb97xv = 0.959274
    freq_ave_wb97xv = (3960.83 + 3859.85) / 2

    shifted_r_cmm_no_fd = r_hbond_no_fd - r_eq_cmm
    shifted_freqs_cmm_no_fd = freqs_hbond_no_fd - freq_ave_cmm

    shifted_r_cmm_with_fd = r_hbond_with_fd - r_eq_cmm
    shifted_freqs_cmm_with_fd = freqs_hbond_with_fd - freq_ave_cmm

    slope_no_fd, intercept_no_fd, _, _, _ = stats.linregress(shifted_r_cmm_no_fd, shifted_freqs_cmm_no_fd)
    slope_with_fd, intercept_with_fd, _, _, _ = stats.linregress(shifted_r_cmm_with_fd, shifted_freqs_cmm_with_fd)
    
    line_x_no_fd = np.linspace(shifted_r_cmm_no_fd.min(), shifted_r_cmm_no_fd.max(), 100)
    line_y_no_fd = slope_no_fd * line_x_no_fd + intercept_no_fd

    line_x_with_fd = np.linspace(shifted_r_cmm_with_fd.min(), shifted_r_cmm_with_fd.max(), 100)
    line_y_with_fd = slope_with_fd * line_x_with_fd + intercept_with_fd
    
    plt.figure(figsize=(10, 6))
    plt.scatter(shifted_r_cmm_no_fd, shifted_freqs_cmm_no_fd, alpha=0.7, color='blue', s=50, label="No FD Morse")
    plt.plot(line_x_no_fd, line_y_no_fd, color='blue', linewidth=2, linestyle="--",
             label=f'Linear fit: y = {slope_no_fd:.2f}x + {intercept_no_fd:.2f}')
    
    plt.scatter(shifted_r_cmm_with_fd, shifted_freqs_cmm_with_fd, alpha=0.7, color='green', s=50, label="With FD Morse")
    plt.plot(line_x_with_fd, line_y_with_fd, color='green', linewidth=2, linestyle="--",
         label=f'Linear fit: y = {slope_with_fd:.2f}x + {intercept_with_fd:.2f}')
    
    plt.xlabel('OH Bond Shift (Å)')
    plt.ylabel('Frequency Shift (cm^-1)')
    plt.title(title)
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print("Linear Regression Results:")
    print(f"Slope No FD: {slope_no_fd:.4f}")
    print(f"Intercept No FD: {intercept_no_fd:.4f}")
    #print(f"R-squared: {r_value**2:.4f}")
    #print(f"Correlation coefficient: {r_value:.4f}")
    #print(f"P-value: {p_value:.4e}")
    #print(f"Standard error: {std_err:.4f}")

In [ ]:
#plot_badger_rule_correlation(all_dists_no_fd, all_freqs_no_fd, all_dists_with_fd, all_freqs_with_fd)